In [17]:
import os 
import torch 
import torch.nn as nn
import torchvision.models  as models 
import torchvision.transforms as transforms 
import torchvision.transforms.functional as TF
from PIL import Image 
import scipy.io

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-->Execution target confirmed: {device}")
if torch.cuda.is_available(): 
    print(f"--> Cloud Hardware Profile: {torch.cuda.get_device_name(0)}") 
    
backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

modules = list(backbone.children())[:-2]
feature_extractor = nn.Sequential(*modules).to(device) 
feature_extractor.eval() 

print("--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully")

-->Execution target confirmed: cuda
--> Cloud Hardware Profile: Tesla T4
--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully


In [19]:
import torch.nn as nn 
import torch.nn.functional as F 

class SaliencyDecoder(nn.Module): 
    def __init__(self): 
        super(SaliencyDecoder, self).__init__() 
        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 1, kernel_size=1)
        
    def forward(self, x): 
        x = F.relu(self.conv1(x)) 
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False) 
        x = self.conv3(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) 
        saliency_map = torch.sigmoid(x)
        return saliency_map

decoder = SaliencyDecoder().to(device)

In [20]:
import torch 

def compute_covariance_manifold(features): 
    B, C, H, W = features.shape 
    N = H * W
    
    reshaped_feats = features.view(B, C, N) 
    mean_feats = reshaped_feats.mean(dim=2, keepdim=True)
    centered_feats = reshaped_feats - mean_feats
    cov_matrices = torch.bmm(centered_feats, centered_feats.transpose(1, 2)) / (N - 1)
    eps = 1e-6 * torch.eye(C, device=features.device).unsqueeze(0).repeat(B, 1, 1)
    spd_matrices = cov_matrices + eps 
    
    return spd_matrices

In [21]:
def matrix_logarithm(spd_matrix): 
    eigenvalues, eigenvectors = torch.linalg.eigh(spd_matrix)
    clipped_evals = torch.clamp(eigenvalues, min=1e-6)
    log_evals = torch.log(clipped_evals) 
    log_diagonal = torch.diag_embed(log_evals)
    matrix_log = torch.bmm(torch.bmm(eigenvectors, log_diagonal), eigenvectors.transpose(1, 2))
    
    return matrix_log

def riemannian_smoothness_loss(features): 
    spd_mats = compute_covariance_manifold(features)
    tangent_projections = matrix_logarithm(spd_mats) 
    batch_mean = tangent_projections.mean(dim=0, keepdim=True) 
    roughness_score = torch.mean((tangent_projections - batch_mean) ** 2)
    
    return roughness_score

In [22]:
def compute_cc(pred_map, gt_map): 
     B, C, H, W = pred_map.shape 
     p = pred_map.view(B, -1) 
     g = gt_map.view(B, -1) 
     
     p_mean = p.mean(dim=1, keepdim=True)
     g_mean = g.mean(dim=1, keepdim=True)
     p_centered = p - p_mean 
     g_centered = g - g_mean 
     
     p_norm = torch.norm(p_centered, p=2, dim=1, keepdim=True)
     g_norm = torch.norm(g_centered, p=2, dim=1, keepdim=True) 
     
     eps = 1e-6 
     cc = torch.sum(p_centered * g_centered, dim=1, keepdim=True) / (p_norm * g_norm + eps)
     
     return cc.mean() 

def compute_sim(pred_map, gt_map): 
    B, C, H, W = pred_map.shape 
    p = pred_map.view(B, -1) 
    g = gt_map.view(B, -1) 
    eps = 1e-6 
    
    p_pdf = p / (p.sum(dim=1, keepdim=True) + eps)
    g_pdf = g / (g.sum(dim=1, keepdim=True)+ eps)
    
    sim = torch.sum(torch.minimum(p_pdf, g_pdf), dim=1)
    
    return sim.mean() 

In [23]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from google.colab import drive

class CAT2000Dataset(Dataset):
    OFFICIAL_CATEGORIES = [
        "Action", "Affective", "Art", "Cartoon", "DigitalArt", 
        "Everyday", "Fractal", "Indoor", "Inverted", "LightNormal", 
        "LineDrawing", "Loud", "LowResolution", "Natural", "Object", 
        "Outdoor", "Photo", "Random", "Social", "Synthetic"
    ]
    
    def __init__(self, root_dir, split='train'):
        self.root_dir = os.path.abspath(root_dir)
        self.stimuli_dir = os.path.join(self.root_dir, "Stimuli")
        self.maps_dir = os.path.join(self.root_dir, "FixationMaps")
        self.cat_to_idx = {cat: idx for idx, cat in enumerate(self.OFFICIAL_CATEGORIES)}
        self.samples = []
        self._index_dataset()
        
    def _index_dataset(self):
        for cat in self.OFFICIAL_CATEGORIES:
            cat_stim = os.path.join(self.stimuli_dir, cat)
            cat_maps = os.path.join(self.maps_dir, cat)
            if os.path.exists(cat_stim) and os.path.exists(cat_maps):
                for file_name in os.listdir(cat_stim):
                    if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(cat_stim, file_name)
                        map_path = os.path.join(cat_maps, file_name)
                        if os.path.exists(map_path):
                            self.samples.append({
                                "image": img_path, "map": map_path, "cat_idx": self.cat_to_idx[cat]
                            })
                            
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image"]).convert("RGB")
        fix_map = Image.open(sample["map"]).convert("L")
        
        # Consistent sizing for the network
        image = TF.resize(image, (224, 224))
        fix_map = TF.resize(fix_map, (224, 224))
        
        # Convert to tensors
        image_tensor = TF.to_tensor(image)
        fix_tensor = TF.to_tensor(fix_map)
        
        # Normalize
        fix_tensor = (fix_tensor - fix_tensor.min()) / (fix_tensor.max() - fix_tensor.min() + 1e-6)
        image_tensor = TF.normalize(image_tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        
        return image_tensor, fix_tensor, sample["cat_idx"]

base_path = "/content/drive/MyDrive/CAT2000/trainSet"
cat2000_dataset = CAT2000Dataset(root_dir=base_path)
production_loader = DataLoader(cat2000_dataset, batch_size=8, shuffle=True)
print(f"--> Dataset initialized with {len(cat2000_dataset)} images.")

--> Dataset initialized with 1200 images.


In [24]:
import torch.optim as optim 

optimizer = optim.Adam(decoder.parameters(), lr=1e-4)
NUM_EPOCHS = 10
ALPHA = 0.05

print("--> Starting training cycle...")

for epoch in range(NUM_EPOCHS): 
    decoder.train()
    epoch_total_loss = 0.0 
    
    for batch_idx, (images, fixation_maps, _) in enumerate(production_loader): 
        optimizer.zero_grad() 
        images = images.to(device)
        fixation_maps = fixation_maps.to(device)
        with torch.no_grad(): 
            features = feature_extractor(images)
        predicted_saliency = decoder(features)
        cc_loss = 1.0 - compute_cc(predicted_saliency, fixation_maps) 
        smoothness_loss = riemannian_smoothness_loss(features)
        total_loss = cc_loss + (ALPHA * smoothness_loss) 
        total_loss.backward() 
        optimizer.step() 
        
        epoch_total_loss += total_loss.item() 
    
    avg_loss = epoch_total_loss / len(production_loader)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Total Loss: {avg_loss:.4f}")

print("\n--> Training Complete. Model is ready for inference.")

--> Starting training cycle...
Epoch [1/10] | Average Total Loss: 0.3566
Epoch [2/10] | Average Total Loss: 0.2142
Epoch [3/10] | Average Total Loss: 0.1572
Epoch [4/10] | Average Total Loss: 0.1320
Epoch [5/10] | Average Total Loss: 0.1180
Epoch [6/10] | Average Total Loss: 0.1066
Epoch [7/10] | Average Total Loss: 0.0971
Epoch [8/10] | Average Total Loss: 0.0874
Epoch [9/10] | Average Total Loss: 0.0831
Epoch [10/10] | Average Total Loss: 0.0780

--> Training Complete. Model is ready for inference.


In [25]:
torch.save(
    decoder.state_dict(),
    "/content/drive/MyDrive/CAT2000/final_decoder.pth"
)

print("Model saved successfully!")

Model saved successfully!


In [31]:
analysis_loader = DataLoader(
    cat2000_dataset,
    batch_size=8,
    shuffle=False
)

In [32]:
import matplotlib.pyplot as plt
import os
import torch

def save_performance_extremes(
    model,
    loader,
    device,
    save_dir="/content/drive/MyDrive/CAT2000/analysis"
):

    os.makedirs(save_dir, exist_ok=True)

    model.eval()
    results = []

    print("Analyzing model performance extremes...")

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    sample_counter = 0

    with torch.no_grad():

        for images, fix_maps, _ in loader:

            images = images.to(device)
            fix_maps = fix_maps.to(device)

            features = feature_extractor(images)
            preds = model(features)

            batch_size = images.shape[0]

            for j in range(batch_size):

                cc_score = compute_cc(
                    preds[j:j+1],
                    fix_maps[j:j+1]
                ).item()

                sample_info = loader.dataset.samples[sample_counter]

                results.append({
                    "index": sample_counter,
                    "score": cc_score,
                    "image": images[j].cpu(),
                    "gt": fix_maps[j].cpu(),
                    "pred": preds[j].cpu(),
                    "sample_info": sample_info
                })

                sample_counter += 1

    results.sort(key=lambda x: x["score"])

    selected = results[:3] + results[-3:]

    for rank, result in enumerate(selected):

        status = "worst" if rank < 3 else "best"

        idx = result["index"]
        score = result["score"]
        img = result["image"]
        gt = result["gt"]
        pred = result["pred"]
        sample_info = result["sample_info"]

        img_display = img * std + mean
        img_display = img_display.clamp(0, 1)

        category = os.path.basename(
            os.path.dirname(sample_info["image"])
        )

        filename = os.path.basename(sample_info["image"])

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(img_display.permute(1, 2, 0))
        plt.title("Input")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(gt.squeeze(), cmap="hot")
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred.squeeze(), cmap="hot")
        plt.title(f"Prediction\nCC={score:.3f}")
        plt.axis("off")

        plt.tight_layout()

        image_path = f"{save_dir}/{status}_{idx}.png"
        metadata_path = f"{save_dir}/{status}_{idx}.txt"

        plt.savefig(image_path, bbox_inches="tight")
        plt.close()

        with open(metadata_path, "w") as f:
            f.write(f"Status: {status}\n")
            f.write(f"Dataset Index: {idx}\n")
            f.write(f"CC Score: {score:.6f}\n")
            f.write(f"Category: {category}\n")
            f.write(f"Filename: {filename}\n\n")
            f.write("Stimulus Image:\n")
            f.write(sample_info["image"] + "\n\n")
            f.write("Fixation Map:\n")
            f.write(sample_info["map"] + "\n")

    print(f"Saved analysis to: {save_dir}")

save_performance_extremes(
    decoder,
    analysis_loader,
    device
)

Analyzing model performance extremes...
Saved analysis to: /content/drive/MyDrive/CAT2000/analysis


In [33]:
import shutil

# Create a zip file containing the entire analysis folder
shutil.make_archive(
    "/content/CAT2000_analysis",
    "zip",
    "/content/drive/MyDrive/CAT2000/analysis"
)

print("Created: /content/CAT2000_analysis.zip")

Created: /content/CAT2000_analysis.zip


In [37]:
import shutil

shutil.copy(
    "/content/CAT2000_analysis.zip",
    "/content/drive/MyDrive/CAT2000_analysis.zip"
)

print("ZIP copied to Google Drive.")

ZIP copied to Google Drive.
